Determines how the tower 33ft wind speed (prop anemometer) and above-building 30ft wind speed (sonic anemometer) compare, using cross-correlation.

The tower's data at a given time, augmented by the lag time at which this cross-correlation peaks, should be the best proxy representation of the live inflow. For example, for peak correlation lag of -4 seconds, the roof-height inflow arriving at the building at time t=10 seconds can be estimated by the tower 13ft wind speed at time t=10-4=6 seconds.

In [ ]:
%load_ext autoreload
%autoreload 2

from pywerfl import loader
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

# Section 1 - bulk comparison of all runs
Studies the correlations between the tower and building wind measurements using all available ingested runs.

The first plot compares the lag time at which a run's peak correlation occurs to the mean wind speed. For a frozen-eddy flow such that the mean wind vector is parallel to the line between the tower and building, we should expect a relationship of $T=\pm D/M$ where $T$ is the lag time, $D$ is the distance between the building and the tower, and $M$ is the mean wind speed. The sign depends on the order cross-correlation is taken. A curve representing this relationship is fit to the data, excluding Run 647 where the wind direction is not well-oriented and the lag time correspondingly does not conform to the curve (this is currently a manual single exclusion, could be extended to further manual exclusions or an automated exclusion/weighting based on wind direction). Using the 16 runs from DesignSafe, a relationship with $D\approx 47.4$ m is found, close to the real distance between the building and tower.

In [ ]:
_TOWER = False # True: x axis uses tower 33ft mean; False: instead x axis is bldg sonic 30 ft mean

lag_limit = 60

run_ids, mean_speeds, peak_lags, peak_corrs = [], [], [], []
for run_id in loader.list_runs():
    run = loader.load_run(run_id)
    tower_speed = run.tower["33ft_wind_speed"].to_numpy()
    sonic_speed = run.sonic["wind_speed"].to_numpy()
    tower_norm = (tower_speed - tower_speed.mean()) / tower_speed.std()
    sonic_norm = (sonic_speed - sonic_speed.mean()) / sonic_speed.std()

    dt = run.tower.index[1] - run.tower.index[0]
    lags = signal.correlation_lags(len(tower_norm), len(sonic_norm)) * dt
    xcorr = signal.correlate(tower_norm, sonic_norm) / len(tower_norm)

    in_window = (lags >= -lag_limit) & (lags <= lag_limit)
    peak_idx = xcorr[in_window].argmax()
    peak_lag = lags[in_window][peak_idx]
    peak_corr = xcorr[in_window][peak_idx]
    print(f"Run {run_id}: peak correlation {peak_corr:.3f} at lag {peak_lag:+.2f} s")

    run_ids.append(run_id)
    mean_speeds.append(tower_speed.mean() if _TOWER else sonic_speed.mean())
    peak_lags.append(peak_lag)
    peak_corrs.append(peak_corr)

mean_speeds, peak_lags, peak_corrs = np.array(mean_speeds), np.array(peak_lags), np.array(peak_corrs)

fig, ax = plt.subplots()
sc = ax.scatter(mean_speeds, peak_lags, c=peak_corrs, cmap="viridis_r")
for run_id, speed, lag in zip(run_ids, mean_speeds, peak_lags):
    ax.annotate(run_id, (speed, lag), fontsize=8)

fit_mask = np.array(run_ids) != "0647"  # excluded: doesn't follow the trend
inv_speed = 1 / mean_speeds[fit_mask]
A = (inv_speed @ peak_lags[fit_mask]) / (inv_speed @ inv_speed)  # least-squares, zero intercept
print(f"Fit (excl. run 647): lag = {A:.4f} / speed")

x_fit = np.linspace(mean_speeds.min(), mean_speeds.max(), 100)
ax.plot(x_fit, A / x_fit, c="k", linestyle="--", linewidth=1, label=f"lag = {A:.2f} m / speed")
ax.legend()

ax.set_xlabel("Mean 33 ft tower wind speed (m/s)" if _TOWER else "Mean 30 ft bldg sonic wind speed (m/s)")
ax.set_ylabel("Lag at peak correlation (s)")
ax.set_title("33 ft tower prop vs 30 ft bldg sonic wind speed,\npeak-correlation lag by run")
fig.colorbar(sc, label="Peak correlation")
plt.show()

This next plot directly compares the mean wind speeds at the tower and building. A unit line (y=x) is overlaid. 

The building (30ft sonic) wind speed does tend to be slightly greater than the tower (33ft propeller) measurement.

In [ ]:
tower_means, sonic_means = [], []
for run_id in run_ids:
    run = loader.load_run(run_id)
    tower_means.append(run.tower["33ft_wind_speed"].mean())
    sonic_means.append(run.sonic["wind_speed"].mean())
tower_means, sonic_means = np.array(tower_means), np.array(sonic_means)

fig, ax = plt.subplots()
ax.scatter(sonic_means, tower_means)
for run_id, x, y in zip(run_ids, sonic_means, tower_means):
    ax.annotate(run_id, (x, y), fontsize=8)

lims = [min(sonic_means.min(), tower_means.min()), max(sonic_means.max(), tower_means.max())]
ax.plot(lims, lims, c="k", linestyle="--", linewidth=1, label="1:1")
ax.legend()

ax.set_xlabel("Mean 30 ft bldg sonic wind speed (m/s)")
ax.set_ylabel("Mean 33 ft tower prop wind speed (m/s)")
ax.set_title("33 ft tower prop vs 30 ft bldg sonic mean wind speed")
plt.show()

In [ ]:
wind_directions = np.array([loader.load_run(rid).metadata["mean_wind_direction_deg"] for rid in run_ids])

fig, ax = plt.subplots()
ax.scatter(wind_directions, peak_corrs)
for run_id, wd, corr in zip(run_ids, wind_directions, peak_corrs):
    ax.annotate(run_id, (wd, corr), fontsize=8)
ax.set_xlabel("Mean wind direction (deg)")
ax.set_ylabel("Peak correlation")
ax.set_title("Peak cross-correlation vs mean wind direction")
plt.show()

# Section 2 - single run visualization / correlations

Compare wind speeds and visualize all lags' cross-correlations for a single run.

In [ ]:
run = loader.load_run(1920) # can replace with any run, must have been ingested

In [ ]:
%matplotlib widget

fig, ax = plt.subplots()
ax.plot(run.tower.index, run.tower["33ft_wind_speed"], c="tab:blue", label="33 ft tower prop", linewidth=0.5)
ax.plot([0,900], [run.tower["33ft_wind_speed"].mean()]*2, c="tab:blue", linestyle="dashed")
ax.plot(run.sonic.index, run.sonic["wind_speed"], c="tab:orange", label="30 ft bldg sonic", linewidth=0.5)
ax.plot([0,900], [run.sonic["wind_speed"].mean()]*2, c="tab:orange", linestyle="dashed")
ax.legend()
plt.show()

In [ ]:
# Cross-correlation between the two wind speed signals. Each is normalized
# (zero mean, unit variance) first so the result is a correlation coefficient
# bounded in [-1, 1] at every lag, rather than a raw, hard-to-interpret magnitude.
tower_speed = run.tower["33ft_wind_speed"].to_numpy()
sonic_speed = run.sonic["wind_speed"].to_numpy()
tower_norm = (tower_speed - tower_speed.mean()) / tower_speed.std()
sonic_norm = (sonic_speed - sonic_speed.mean()) / sonic_speed.std()

dt = run.tower.index[1] - run.tower.index[0]
lags = signal.correlation_lags(len(tower_norm), len(sonic_norm)) * dt
xcorr = signal.correlate(tower_norm, sonic_norm) / len(tower_norm)

fig, ax = plt.subplots()
ax.plot(lags, xcorr, linewidth=0.5)
ax.axvline(0, c="k", linewidth=0.5, linestyle="--")
ax.set_xlim(-30, 30)
ax.set_xlabel("Lag (s)")
ax.set_ylabel("Cross-correlation")
ax.set_title("33 ft tower prop vs 30 ft bldg sonic wind speed")
plt.show()